# Korea FCFF Valuation from DART **v1.1**

DART 수집 데이터 + 기존 매출 예측 기반 self-contained FCFF DCF.

**v1.0 → v1.1 수정:**
1. D&A/무형상각/CapEx account_nm 패턴 보강 (앞머리 고정 제거, 합산계정 이중계상 방지)
2. ΔNWC 시드를 v8 방식(γ × 마지막 실측 매출)으로 변경 — 첫 분기 ΔNWC 스파이크 제거
3. **실적↔예측 연속성 가드** 신설: 최근 4분기 실적 중앙값 대비 예측 첫 분기가 0.6~1.4배 밖이면 즉시 중단
4. 진단 셀 신설: `check_field_mapping` / `diagnose_accounts` — 매핑 실패 필드의 실제 계정명 확인용

In [1]:
# ═══════════════════════════════════════════════════════════════
#  ★ 입력 변수 — 이 셀만 수정하세요
# ═══════════════════════════════════════════════════════════════
import os
from pathlib import Path

# ① 평가 대상 (6자리 코드, 'A' 접두어 유무 모두 허용)
EXPORT_TICKERS = [
    "005930",      # 삼성전자
    # "000660",    # SK하이닉스
]

# ② DB
DB_INFO = {
    "host": "192.168.0.230",
    "port": 3307,
    "user": "stox7412",
    "password": "Apt106503!~",
    "database": "investar",
}
TABLE_DART_FS   = "korea_fs_data_from_DART_V2"       # DART 재무 long 테이블
TABLE_FORECAST  = "korea_revenue_forecast_result"    # 기존 매출 예측 (천원 단위)
TABLE_MARKETCAP = "ks_listed_company_daily_marketcap"

# ③ 모델 파라미터 (v8 기본값 계승)
FORECAST_HORIZON = 8          # 예측 분기 수
MIN_HISTORY      = 16         # 계수 추정 최소 분기
WINSORIZE_LIMITS = (0.05, 0.95)
OLS_MIN_R2       = 0.30
OLS_MIN_SAMPLES  = 16
GDP_GROWTH       = 0.04       # terminal g 상한
TAX_FALLBACK     = 0.22       # 실효세율 추정 실패 시

# ④ WACC
RF               = 0.035      # 무위험이자율 (10Y 국고채 수동 입력)
ERP              = 0.07       # Damodaran 한국 ERP
RD_DEFAULT       = 0.045      # 부채비용 fallback
BETA_DEFAULT     = 1.0        # 베타 산출 실패 시
BETA_OVERRIDE    = {}         # 예: {"005930": 0.95}
WACC_FLOOR_ABS   = 0.05       # WACC 하한 = max(이 값, Rf+1%)
WACC_CAP         = 0.20
MIN_TV_SPREAD    = 0.02       # WACC − g_term 최소 스프레드
TV_FCFF_MULT_CAP = 35.0       # TV / 연간 FCFF 배수 상한

# ⑤ 단위
MARKETCAP_UNIT_MULTIPLIER = 1_000_000   # 시총 테이블: 백만원 → 원
FORECAST_UNIT_MULTIPLIER  = 1_000       # 예측 테이블: 천원 → 원
# DART 금액은 원 단위 그대로 사용 (변환 없음)

VERBOSE = True

print("[OK] 설정 완료 —", EXPORT_TICKERS)


[OK] 설정 완료 — ['005930']


## Cell 2 · DART → 표준 wide 변환

In [2]:
# ═══════════════════════════════════════════════════════════════
#  DART long → 표준 wide 변환 모듈
#  - account_id 우선 매칭, 실패 시 account_nm 정규식 fallback
#  - IS/CIS: 누적 vs 3개월 자동 감지 후 분기화 (Q4 = FY − 3개분기)
#  - CF: 항상 누적으로 간주 → 차분 (Q2=H1−Q1, Q3=Q3−H1, Q4=FY−Q3)
#  - BS: 시점 잔액 그대로
# ═══════════════════════════════════════════════════════════════
import re
import numpy as np
import pandas as pd
import pymysql
from datetime import datetime


def log(tag, msg):
    print(f"[{datetime.now():%H:%M:%S}][{tag}] {msg}", flush=True)


def norm_ticker(t: str) -> str:
    """'A005930' → '005930'"""
    t = str(t).strip().upper()
    return t[1:].zfill(6) if t.startswith("A") else t.zfill(6)


def to_dg_ticker(t: str) -> str:
    """'005930' → 'A005930' (forecast 테이블용)"""
    return "A" + norm_ticker(t)


# ───────────────────────────────────────────────────────────────
#  표준 필드 매핑 정의
#    ids : account_id 후보 (정확 일치, 접두어 ifrs_/ifrs-full_ 모두 등록)
#    nm  : account_nm 정규식 후보 (앞에 있을수록 우선)
#    sj  : 허용 재무제표 (앞에 있을수록 우선; IS 우선, 없으면 CIS)
#    agg : 'pick'=대표 계정 1개 선택(중복합산 방지) / 'sum'=매칭 계정 전부 합산
# ───────────────────────────────────────────────────────────────
def _ids(*stems):
    out = []
    for s in stems:
        out += [f"ifrs_{s}", f"ifrs-full_{s}"]
    return out


FIELD_MAP = {
    # ── 손익 (flow) ──
    "revenue": dict(
        ids=_ids("Revenue") + ["dart_Revenue"],
        nm=[r"^매출액$", r"^매출$", r"^수익\(매출액\)$", r"^영업수익$"],
        sj=["IS", "CIS"], agg="pick"),
    "operating_income": dict(
        ids=["dart_OperatingIncomeLoss"] + _ids("OperatingIncomeLoss"),
        nm=[r"^영업이익", r"^영업손익"],
        sj=["IS", "CIS"], agg="pick"),
    "pretax_income": dict(
        ids=_ids("ProfitLossBeforeTax"),
        nm=[r"법인세비용차감전", r"^세전.*이익"],
        sj=["IS", "CIS"], agg="pick"),
    "tax_expense": dict(
        ids=_ids("IncomeTaxExpenseContinuingOperations", "IncomeTaxExpense"),
        nm=[r"^법인세비용"],
        sj=["IS", "CIS"], agg="pick"),
    "interest_expense": dict(
        ids=_ids("FinanceCosts") + ["dart_InterestExpenseFinanceExpense"],
        nm=[r"^이자비용", r"^금융비용"],
        sj=["IS", "CIS"], agg="pick"),

    # ── 현금흐름 (flow, 누적) ──
    "da_cf": dict(
        ids=["dart_AdjustmentsForDepreciationExpense"] +
            _ids("AdjustmentsForDepreciationExpense",
                 "AdjustmentsForDepreciationAndAmortisationExpense",
                 "DepreciationAndAmortisationExpense"),
        nm=[r"감가상각비와\s*무형자산상각", r"감가상각비\s*및\s*상각",
            r"감가상각"],                    # 앞머리 고정 제거 — "유형자산 감가상각비" 등 대응
        sj=["CF"], agg="pick"),
    "intangible_amort_cf": dict(
        ids=["dart_AmortisationExpense"] +
            _ids("AdjustmentsForAmortisationExpense", "AmortisationExpense"),
        nm=[r"^(?!.*감가상각).*무형자산\s*상각"],   # 합산계정(감가상각비와 무형자산상각비) 제외 — da_cf 와 이중계상 방지
        sj=["CF"], agg="pick"),
    "capex_tangible": dict(
        ids=_ids("PurchaseOfPropertyPlantAndEquipmentClassifiedAsInvestingActivities",
                 "PurchaseOfPropertyPlantAndEquipment"),
        nm=[r"유형자산의\s*취득", r"유형자산의\s*증가", r"유형자산\s*취득",
            r"토지.*취득|건설중인자산.*(취득|증가)"],
        sj=["CF"], agg="pick"),
    "capex_intangible": dict(
        ids=_ids("PurchaseOfIntangibleAssetsClassifiedAsInvestingActivities",
                 "PurchaseOfIntangibleAssets"),
        nm=[r"무형자산의\s*취득", r"무형자산의\s*증가", r"무형자산\s*취득"],
        sj=["CF"], agg="pick"),

    # ── 재무상태 (stock) ──
    "receivables": dict(
        ids=_ids("TradeAndOtherCurrentReceivables", "CurrentTradeReceivables"),
        nm=[r"^매출채권$", r"^매출채권\s*및", r"^매출채권과"],
        sj=["BS"], agg="pick"),
    "inventories": dict(
        ids=_ids("Inventories"),
        nm=[r"^재고자산"],
        sj=["BS"], agg="pick"),
    "prepaid_expenses": dict(
        ids=[], nm=[r"^선급비용"], sj=["BS"], agg="pick"),
    "payables": dict(
        ids=_ids("TradeAndOtherCurrentPayables", "CurrentTradePayables"),
        nm=[r"^매입채무$", r"^매입채무\s*및", r"^매입채무와"],
        sj=["BS"], agg="pick"),
    "accrued_expenses": dict(
        ids=[], nm=[r"^미지급비용"], sj=["BS"], agg="pick"),
    "other_payables": dict(
        ids=[], nm=[r"^미지급금"], sj=["BS"], agg="pick"),
    "advances_received": dict(
        ids=[], nm=[r"^선수금"], sj=["BS"], agg="pick"),
    "contract_liabilities": dict(
        ids=_ids("ContractLiabilities"),
        nm=[r"^계약부채"], sj=["BS"], agg="pick"),

    "short_term_debt": dict(
        ids=["dart_ShortTermBorrowings"] + _ids("ShorttermBorrowings"),
        nm=[r"^단기차입금"], sj=["BS"], agg="pick"),
    "current_lt_debt": dict(
        ids=[], nm=[r"^유동성장기부채", r"^유동성장기차입금", r"^유동성사채"],
        sj=["BS"], agg="sum"),
    "bonds": dict(
        ids=[], nm=[r"^사채$", r"^사채\("], sj=["BS"], agg="pick"),
    "long_term_debt": dict(
        ids=["dart_LongTermBorrowingsGross"],
        nm=[r"^장기차입금"], sj=["BS"], agg="pick"),
    "lease_liab": dict(
        ids=_ids("LeaseLiabilities", "CurrentLeaseLiabilities",
                 "NoncurrentLeaseLiabilities"),
        nm=[r"리스부채"], sj=["BS"], agg="sum"),

    "cash": dict(
        ids=_ids("CashAndCashEquivalents"),
        nm=[r"^현금및현금성자산"], sj=["BS"], agg="pick"),
    "short_term_invest": dict(
        ids=["dart_ShortTermDepositsNotClassifiedAsCashEquivalents"],
        nm=[r"^단기금융상품", r"^단기투자자산"], sj=["BS"], agg="pick"),
    "total_equity": dict(
        ids=_ids("Equity"),
        nm=[r"^자본총계"], sj=["BS"], agg="pick"),
    "total_assets": dict(
        ids=_ids("Assets"),
        nm=[r"^자산총계"], sj=["BS"], agg="pick"),
}

QUARTER_ORDER = {"Q1": 1, "H1": 2, "Q3": 3, "FY": 4}
FLOW_SJ  = {"IS", "CIS", "CF"}


def _load_ticker_long(ticker: str, db_info: dict, table: str) -> pd.DataFrame:
    conn = pymysql.connect(**db_info, charset="utf8mb4")
    try:
        df = pd.read_sql(
            f"""SELECT bsns_year, quarter, sj_div, account_id, account_nm,
                       thstrm_amount, report_date
                FROM {table} WHERE ticker = %s""",
            conn, params=[norm_ticker(ticker)])
    finally:
        conn.close()
    df["thstrm_amount"] = pd.to_numeric(df["thstrm_amount"], errors="coerce")
    df["report_date"] = pd.to_datetime(df["report_date"])
    return df


def _match_field(df: pd.DataFrame, spec: dict) -> pd.DataFrame:
    """필드 정의(spec)에 맞는 행 선택. 반환: (bsns_year, quarter) 별 단일 값."""
    sub = df[df["sj_div"].isin(spec["sj"])].copy()
    if sub.empty:
        return pd.DataFrame()

    # 1) account_id 정확 일치
    hit = sub[sub["account_id"].isin(spec["ids"])] if spec["ids"] else pd.DataFrame()

    # 2) 실패 시 account_nm 정규식 (패턴 순서 = 우선순위)
    if hit.empty and spec["nm"]:
        for pat in spec["nm"]:
            m = sub[sub["account_nm"].astype(str).str.strip()
                       .str.contains(pat, regex=True, na=False)]
            if not m.empty:
                hit = m
                break
    if hit.empty:
        return pd.DataFrame()

    # sj_div 우선순위 (IS > CIS 등): 상위 sj에 데이터가 있으면 그것만
    for sj in spec["sj"]:
        h2 = hit[hit["sj_div"] == sj]
        if not h2.empty:
            hit = h2
            break

    if spec["agg"] == "sum":
        # 서로 다른 account_id 를 (연도,분기)별 합산 (리스부채 유동+비유동 등)
        out = (hit.groupby(["bsns_year", "quarter"], as_index=False)
                  ["thstrm_amount"].sum(min_count=1))
    else:
        # 'pick': 기간 커버리지가 가장 넓은 대표 account_id 1개만 사용 (중복합산 방지)
        cov = hit.groupby("account_id")["bsns_year"].count().sort_values(ascending=False)
        best = cov.index[0]
        out = hit[hit["account_id"] == best][
            ["bsns_year", "quarter", "thstrm_amount"]].copy()
        # 같은 (연도,분기) 중복 시 첫 행
        out = out.drop_duplicates(subset=["bsns_year", "quarter"])
    return out


def _detect_cumulative(pivot: pd.DataFrame) -> bool:
    """
    IS 계열 flow 가 누적인지 3개월치인지 자동 감지.
    (Q1+H1+Q3)/FY 중앙값: 3개월치 ≈ 0.75, 누적 ≈ 1.5 → 임계 1.1
    """
    ratios = []
    for y, row in pivot.iterrows():
        if all(pd.notnull(row.get(q)) for q in ("Q1", "H1", "Q3", "FY")) \
                and row["FY"] not in (0, None):
            ratios.append((row["Q1"] + row["H1"] + row["Q3"]) / row["FY"])
    if not ratios:
        return False   # 판단 불가 → 3개월치 가정 (보수적)
    return float(np.median(ratios)) > 1.1


def _flow_to_quarterly(series_df: pd.DataFrame, force_cumulative: bool = None):
    """
    flow 항목 (연도,분기,값) → 분기화 값 dict {(year,'Qn'): value}.
    force_cumulative: None=자동감지, True=누적 차분, False=3개월치 취급
    반환: (dict, cumulative여부)
    """
    pivot = series_df.pivot_table(index="bsns_year", columns="quarter",
                                  values="thstrm_amount", aggfunc="first")
    cum = _detect_cumulative(pivot) if force_cumulative is None else force_cumulative

    out = {}
    for y, row in pivot.iterrows():
        q1, h1, q3, fy = (row.get("Q1"), row.get("H1"),
                          row.get("Q3"), row.get("FY"))
        if cum:
            out[(y, "Q1")] = q1
            out[(y, "Q2")] = h1 - q1 if pd.notnull(h1) and pd.notnull(q1) else np.nan
            out[(y, "Q3")] = q3 - h1 if pd.notnull(q3) and pd.notnull(h1) else np.nan
            out[(y, "Q4")] = fy - q3 if pd.notnull(fy) and pd.notnull(q3) else np.nan
        else:
            out[(y, "Q1")] = q1
            out[(y, "Q2")] = h1
            out[(y, "Q3")] = q3
            if all(pd.notnull(v) for v in (fy, q1, h1, q3)):
                out[(y, "Q4")] = fy - (q1 + h1 + q3)
            else:
                out[(y, "Q4")] = np.nan
    return out, cum


_QDATE = {"Q1": "-03-31", "Q2": "-06-30", "Q3": "-09-30", "Q4": "-12-31"}


def load_dart_financials_wide(ticker: str, db_info: dict,
                              table_name: str = None,
                              item_keys=None, fillna_zero: bool = False,
                              verbose: bool = False) -> pd.DataFrame:
    """
    DART long 테이블 → 분기 wide DataFrame (index=분기말 date, 단위=원).
    기존 load_korea_financials_wide 와 동일한 사용 패턴.
    """
    table_name = table_name or TABLE_DART_FS
    raw = _load_ticker_long(ticker, db_info, table_name)
    if raw.empty:
        return pd.DataFrame()

    fields = item_keys or list(FIELD_MAP.keys())
    col_data, cum_info = {}, {}

    for f in fields:
        spec = FIELD_MAP[f]
        sel = _match_field(raw, spec)
        if sel.empty:
            continue
        if spec["sj"][0] in FLOW_SJ:
            force = True if spec["sj"] == ["CF"] else None   # CF는 항상 누적
            qvals, cum = _flow_to_quarterly(sel, force_cumulative=force)
            cum_info[f] = cum
        else:  # BS: 시점 잔액, H1→Q2 라벨만 변경
            qvals = {}
            for _, r in sel.iterrows():
                q = {"Q1": "Q1", "H1": "Q2", "Q3": "Q3", "FY": "Q4"}[r["quarter"]]
                qvals[(int(r["bsns_year"]), q)] = r["thstrm_amount"]
        col_data[f] = qvals

    if not col_data:
        return pd.DataFrame()

    all_keys = sorted({k for v in col_data.values() for k in v})
    idx = pd.to_datetime([f"{y}{_QDATE[q]}" for y, q in all_keys])
    wide = pd.DataFrame(
        {f: [col_data[f].get(k, np.nan) for k in all_keys] for f in col_data},
        index=idx).sort_index()

    if fillna_zero:
        wide = wide.fillna(0.0)

    if verbose:
        cum_flows = [f for f, c in cum_info.items() if c]
        log(norm_ticker(ticker),
            f"wide shape={wide.shape} 기간={wide.index.min().date()}~{wide.index.max().date()}"
            + (f"  누적차분 적용: {cum_flows}" if cum_flows else ""))
    return wide


print("[OK] DART wide 변환 모듈 로드 완료")


[OK] DART wide 변환 모듈 로드 완료


## Cell 3 · 검증 + 보조 로더

In [3]:
# ═══════════════════════════════════════════════════════════════
#  데이터 검증 + 보조 로더 (매출 예측 · 시가총액)
# ═══════════════════════════════════════════════════════════════
import pymysql
import numpy as np
import pandas as pd


def validate_dart_wide(ticker: str):
    """한 종목의 wide 변환 결과를 요약 출력 (모델 실행 전 육안 점검용)."""
    w = load_dart_financials_wide(ticker, DB_INFO, TABLE_DART_FS, verbose=True)
    if w.empty:
        print(f"❌ {ticker}: DART 데이터 없음")
        return None
    print(f"\n[필드 커버리지] (비결측 분기 수 / 전체 {len(w)}분기)")
    cov = w.notna().sum().sort_values(ascending=False)
    print(cov.to_string())

    # 매출 분기화 정합성: 연도별 4분기 합 vs 재계산 FY
    if "revenue" in w.columns:
        rev = w["revenue"].dropna()
        yearly = rev.groupby(rev.index.year).agg(["count", "sum"])
        full = yearly[yearly["count"] == 4]
        print(f"\n[매출 분기화 결과 — 4분기 완비 연도 {len(full)}개]")
        print((full["sum"] / 1e12).round(2).rename("연매출(조원)").to_string())

    print("\n[최근 6분기 주요 항목 (조원)]")
    key = [c for c in ["revenue", "operating_income", "da_cf",
                       "capex_tangible", "receivables", "inventories",
                       "payables", "total_equity", "cash"] if c in w.columns]
    print((w[key].tail(6) / 1e12).round(3).to_string())
    return w


def load_revenue_forecast(ticker: str, horizon: int = FORECAST_HORIZON):
    """
    기존 korea_revenue_forecast_result 에서 최신 실행 버전 로드.
    (v1.1 patch 로직 이식: updated_at 우선, 분기수·연속성·상수 3중 검증)
    반환: (pd.Series[원 단위], model_name, run_date)
    """
    dg = to_dg_ticker(ticker)
    conn = pymysql.connect(**DB_INFO, charset="utf8mb4")
    try:
        cur = conn.cursor()
        cur.execute("""SELECT COLUMN_NAME FROM information_schema.COLUMNS
                       WHERE TABLE_SCHEMA=%s AND TABLE_NAME=%s""",
                    (DB_INFO["database"], TABLE_FORECAST))
        cols = {r[0].lower() for r in cur.fetchall()}
        rc = "updated_at" if "updated_at" in cols else "created_at"

        for model in ("Ensemble", "SARIMA", "ETS", "Theta"):
            cur.execute(
                f"""SELECT MAX(DATE({rc})) FROM {TABLE_FORECAST}
                    WHERE ticker=%s AND (indicator=%s OR indicator LIKE %s)""",
                (dg, model, f"%\\_{model}"))
            row = cur.fetchone()
            if not row or row[0] is None:
                continue
            run_date = str(row[0])
            df = pd.read_sql(
                f"""SELECT date, value, {rc} AS run_ts FROM {TABLE_FORECAST}
                    WHERE ticker=%s AND (indicator=%s OR indicator LIKE %s)
                      AND DATE({rc})=%s ORDER BY date""",
                conn, params=[dg, model, f"%\\_{model}", run_date])
            if df.empty:
                continue

            df["date"] = pd.to_datetime(df["date"])
            df["run_ts"] = pd.to_datetime(df["run_ts"])
            df = (df.sort_values(["date", "run_ts"])
                    .drop_duplicates(subset=["date"], keep="last"))
            fc = df.set_index("date")["value"].astype(float).sort_index()

            if len(fc) < horizon:
                raise ValueError(f"[{ticker}] forecast {len(fc)}분기 < horizon {horizon}")
            fc = fc.iloc[:horizon]
            per = fc.index.to_period("Q")
            if ((per[1:].astype("int64") - per[:-1].astype("int64")) != 1).any():
                raise ValueError(f"[{ticker}] forecast 분기 불연속 — 재예측 필요")
            if fc.nunique() == 1:
                raise ValueError(f"[{ticker}] forecast 상수 시계열 — 재예측 필요")

            return fc * FORECAST_UNIT_MULTIPLIER, model, run_date
    finally:
        conn.close()
    raise ValueError(f"[{ticker}] 매출 forecast 없음 — Korea_revenue_forecast 먼저 실행")


def load_marketcap_latest(ticker: str):
    """
    ks_listed_company_daily_marketcap 에서 최신 시가총액 로드.
    컬럼명이 확실치 않아 introspection 으로 ticker/date/marketcap 컬럼 자동 탐색.
    반환: (시가총액 [원], 기준일) — 실패 시 (None, None)
    """
    conn = pymysql.connect(**DB_INFO, charset="utf8mb4")
    try:
        cur = conn.cursor()
        cur.execute("""SELECT COLUMN_NAME FROM information_schema.COLUMNS
                       WHERE TABLE_SCHEMA=%s AND TABLE_NAME=%s""",
                    (DB_INFO["database"], TABLE_MARKETCAP))
        cols = [r[0] for r in cur.fetchall()]
        low = {c.lower(): c for c in cols}

        tcol = next((low[k] for k in ("ticker", "code", "stock_code", "종목코드")
                     if k in low), None)
        dcol = next((low[k] for k in ("date", "base_date", "일자", "기준일")
                     if k in low), None)
        mcol = next((low[k] for k in ("marketcap", "market_cap", "marcap",
                                      "시가총액") if k in low), None)
        if not (tcol and dcol and mcol):
            print(f"⚠️ 시총 테이블 컬럼 자동 탐색 실패 (cols={cols[:8]}...)")
            return None, None

        for key in (norm_ticker(ticker), to_dg_ticker(ticker)):
            cur.execute(
                f"""SELECT `{mcol}`, `{dcol}` FROM {TABLE_MARKETCAP}
                    WHERE `{tcol}`=%s ORDER BY `{dcol}` DESC LIMIT 1""", (key,))
            row = cur.fetchone()
            if row and row[0] is not None:
                return float(row[0]) * MARKETCAP_UNIT_MULTIPLIER, row[1]
        return None, None
    finally:
        conn.close()


print("[OK] 검증·보조 로더 준비 완료")
# 빠른 점검:
_ = validate_dart_wide(EXPORT_TICKERS[0])


[OK] 검증·보조 로더 준비 완료


C:\Users\82108\AppData\Local\Temp\ipykernel_25268\2775365695.py:155: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


[22:48:09][005930] wide shape=(44, 22) 기간=2015-03-31~2025-12-31  누적차분 적용: ['capex_tangible', 'capex_intangible']

[필드 커버리지] (비결측 분기 수 / 전체 44분기)
operating_income     39
long_term_debt       31
total_assets         25
inventories          25
cash                 25
short_term_invest    24
bonds                24
revenue              24
capex_intangible     24
capex_tangible       24
tax_expense          24
pretax_income        24
short_term_debt      17
current_lt_debt      16
total_equity         16
receivables           8
payables              8
other_payables        8
interest_expense      8
accrued_expenses      8
prepaid_expenses      5
advances_received     5

[매출 분기화 결과 — 4분기 완비 연도 5개]
2020    236.81
2021    279.60
2022    302.23
2023    258.94
2024    300.87

[최근 6분기 주요 항목 (조원)]
            revenue  operating_income  capex_tangible  receivables  inventories  payables  total_equity    cash
2024-09-30   79.099             9.183          10.959       44.692       53.357    12.862  

## Cell 4 · 진단 (매핑 확인)

모델 실행 전 `check_field_mapping` 결과에서 ❌ 필드를 확인하세요.
❌가 있으면 `diagnose_accounts(ticker, sj_div=..., keyword=...)` 로 실제 계정명을 찾아
Cell 2 FIELD_MAP 의 nm 패턴에 추가 후 Cell 2부터 재실행.

In [4]:
# ═══════════════════════════════════════════════════════════════
#  진단 — 특정 종목의 DART 계정 목록 덤프 (매핑 보강용)
#  da=0 처럼 필드가 비면 이 셀을 실행해 실제 account_id/account_nm 을 확인하고
#  Cell 2 의 FIELD_MAP 에 패턴을 추가하세요.
# ═══════════════════════════════════════════════════════════════
import pandas as pd


def diagnose_accounts(ticker: str, sj_div: str = None, keyword: str = None,
                      top: int = 60):
    """
    종목의 (sj_div, account_id, account_nm) 별 커버리지·최근 FY 금액 요약.
    sj_div : 'CF','BS','IS','CIS' 필터 (None=전체)
    keyword: account_nm 포함 검색어 (예: '상각', '취득', '차입')
    """
    raw = _load_ticker_long(ticker, DB_INFO, TABLE_DART_FS)
    if raw.empty:
        print(f"❌ {ticker}: 데이터 없음")
        return None
    df = raw.copy()
    if sj_div:
        df = df[df["sj_div"] == sj_div]
    if keyword:
        df = df[df["account_nm"].astype(str).str.contains(keyword, na=False)]

    last_fy = df[df["quarter"] == "FY"]["bsns_year"].max()
    fy_amt = (df[(df["quarter"] == "FY") & (df["bsns_year"] == last_fy)]
              .set_index(["sj_div", "account_id", "account_nm"])["thstrm_amount"])

    g = (df.groupby(["sj_div", "account_id", "account_nm"])
           .agg(n_periods=("bsns_year", "count"),
                yr_min=("bsns_year", "min"), yr_max=("bsns_year", "max"))
           .sort_values("n_periods", ascending=False))
    g["last_FY_억원"] = (fy_amt.reindex(g.index) / 1e8).round(0)
    print(f"[{norm_ticker(ticker)}] sj={sj_div or 'ALL'} kw={keyword or '-'} "
          f"— 계정 {len(g)}개 (최근 FY={last_fy})")
    with pd.option_context("display.max_rows", top, "display.width", 200,
                           "display.max_colwidth", 45):
        print(g.head(top).to_string())
    return g


def check_field_mapping(ticker: str):
    """FIELD_MAP 각 필드가 이 종목에서 어떤 계정으로 resolve 되는지 일람."""
    raw = _load_ticker_long(ticker, DB_INFO, TABLE_DART_FS)
    print(f"[{norm_ticker(ticker)}] 필드 → 매칭 계정")
    for f, spec in FIELD_MAP.items():
        sel_rows = raw[raw["sj_div"].isin(spec["sj"])]
        hit = sel_rows[sel_rows["account_id"].isin(spec["ids"])] if spec["ids"] else sel_rows.iloc[0:0]
        via = "id"
        if hit.empty and spec["nm"]:
            for pat in spec["nm"]:
                m = sel_rows[sel_rows["account_nm"].astype(str).str.strip()
                             .str.contains(pat, regex=True, na=False)]
                if not m.empty:
                    hit, via = m, f"nm:{pat}"
                    break
        if hit.empty:
            print(f"  ❌ {f:<22} 매칭 없음")
        else:
            names = hit.groupby(["account_id", "account_nm"]).size() \
                       .sort_values(ascending=False)
            best = names.index[0]
            print(f"  ✅ {f:<22} [{via}] {best[0]} / {best[1]} "
                  f"({names.iloc[0]}기간{', 후보 '+str(len(names))+'개' if len(names)>1 else ''})")


# 실행 예: D&A가 0으로 나온 종목의 CF 계정 확인
check_field_mapping(EXPORT_TICKERS[0])
print()
diagnose_accounts(EXPORT_TICKERS[0], sj_div="CF", keyword="상각")


[005930] 필드 → 매칭 계정
  ✅ revenue                [id] ifrs-full_Revenue / 수익(매출액) (17기간, 후보 4개)
  ✅ operating_income       [id] dart_OperatingIncomeLoss / 영업이익 (23기간, 후보 2개)
  ✅ pretax_income          [id] ifrs-full_ProfitLossBeforeTax / 법인세비용차감전순이익(손실) (20기간, 후보 3개)
  ✅ tax_expense            [id] ifrs-full_IncomeTaxExpenseContinuingOperations / 법인세비용 (19기간, 후보 3개)
  ✅ interest_expense       [id] ifrs-full_FinanceCosts / 금융비용 (9기간, 후보 2개)
  ❌ da_cf                  매칭 없음
  ❌ intangible_amort_cf    매칭 없음
  ✅ capex_tangible         [id] ifrs-full_PurchaseOfPropertyPlantAndEquipmentClassifiedAsInvestingActivities / 유형자산의 취득 (25기간, 후보 2개)
  ✅ capex_intangible       [id] ifrs-full_PurchaseOfIntangibleAssetsClassifiedAsInvestingActivities / 무형자산의 취득 (25기간, 후보 2개)
  ✅ receivables            [id] ifrs-full_CurrentTradeReceivables / 매출채권 (8기간)
  ✅ inventories            [id] ifrs-full_Inventories / 재고자산 (25기간, 후보 2개)
  ✅ prepaid_expenses       [nm:^선급비용] ifrs-full_CurrentPrepaidExpenses / 선급비용 (

C:\Users\82108\AppData\Local\Temp\ipykernel_25268\2775365695.py:155: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(
C:\Users\82108\AppData\Local\Temp\ipykernel_25268\2775365695.py:155: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


[005930] sj=CF kw=상각 — 계정 0개 (최근 FY=nan)
Empty DataFrame
Columns: [n_periods, yr_min, yr_max, last_FY_억원]
Index: []


,,,n_periods,yr_min,yr_max,last_FY_억원
sj_div,account_id,account_nm,,,,


## Cell 5 · DartFCFFModel

In [5]:
# ═══════════════════════════════════════════════════════════════
#  DartFCFFModel — DART 기반 Sales-driven FCFF DCF (v8 로직 계승)
#    FCFF_q = NOPAT + D&A − CapEx − ΔNWC
#    계수(OPM, D&A/S, CapEx/S, NWC/S)는 OLS → 부적합 시 winsorized median
#    v8 가드: Rd≥Rf · WACC≥max(5%,Rf+1%) · g_term≤min(GDP,Rf)
#             · TV 스프레드 ≥2% · TV/연FCFF ≤35배
# ═══════════════════════════════════════════════════════════════
import numpy as np
import pandas as pd
from scipy import stats


def _winsorize(s: pd.Series, limits=WINSORIZE_LIMITS) -> pd.Series:
    s = s.dropna()
    if len(s) < 4:
        return s
    return s.clip(s.quantile(limits[0]), s.quantile(limits[1]))


def _ols_ratio(x: pd.Series, y: pd.Series):
    mask = x.notna() & y.notna() & (x != 0)
    if mask.sum() < OLS_MIN_SAMPLES:
        return np.nan, -1.0, int(mask.sum())
    slope, _, r, _, _ = stats.linregress(x[mask], y[mask])
    return float(slope), float(r ** 2), int(mask.sum())


class DartFCFFModel:

    NWC_ASSETS = ["receivables", "inventories", "prepaid_expenses"]
    NWC_LIABS  = ["payables", "accrued_expenses", "other_payables",
                  "advances_received", "contract_liabilities"]
    DEBT_KEYS  = ["short_term_debt", "current_lt_debt", "bonds",
                  "long_term_debt", "lease_liab"]
    CASH_KEYS  = ["cash", "short_term_invest"]

    def __init__(self, ticker: str, verbose: bool = VERBOSE):
        self.ticker = norm_ticker(ticker)
        self.verbose = verbose
        self.notes = []

    def _log(self, msg):
        if self.verbose:
            log(self.ticker, msg)

    def _note(self, msg):
        self.notes.append(msg)
        self._log(msg)

    # ── 1. 데이터 ────────────────────────────────────────────
    def load(self):
        self.wide = load_dart_financials_wide(
            self.ticker, DB_INFO, TABLE_DART_FS, verbose=self.verbose)
        if self.wide.empty:
            raise ValueError(f"[{self.ticker}] DART 재무 데이터 없음")
        if "revenue" not in self.wide.columns:
            raise ValueError(f"[{self.ticker}] 매출 데이터 없음")
        if "operating_income" not in self.wide.columns:
            raise ValueError(f"[{self.ticker}] 영업이익 없음 — 가치평가 불가")

        self.sales = self.wide["revenue"].dropna()
        if len(self.sales) < MIN_HISTORY:
            raise ValueError(
                f"[{self.ticker}] 매출 {len(self.sales)}분기 < 최소 {MIN_HISTORY}분기")

        self.fc_sales, self.fc_model, self.fc_date = \
            load_revenue_forecast(self.ticker, FORECAST_HORIZON)
        self._log(f"매출 actual={len(self.sales)}Q forecast={len(self.fc_sales)}Q "
                  f"({self.fc_model}, {self.fc_date})")

        # ★ 실적↔예측 연속성 가드: 단위 오류·깨진 예측·분기화 오류를 조기 차단
        last4_med = float(self.sales.iloc[-4:].median())
        first_fc  = float(self.fc_sales.iloc[0])
        ratio = first_fc / last4_med if last4_med > 0 else np.nan
        self._log("실적 vs 예측 연결 구간:")
        if self.verbose:
            tail = (self.sales.iloc[-4:] / 1e12).round(2)
            head = (self.fc_sales.iloc[:4] / 1e12).round(2)
            print("  [actual 조원]", dict(zip(tail.index.strftime("%Y-%m"), tail.values)))
            print("  [forecast 조원]", dict(zip(head.index.strftime("%Y-%m"), head.values)))
        if not (0.6 <= ratio <= 1.4):
            raise ValueError(
                f"[{self.ticker}] 실적-예측 단절: 최근 4분기 중앙값 {last4_med/1e12:.1f}조 vs "
                f"예측 첫 분기 {first_fc/1e12:.1f}조 (배율 {ratio:.2f}). "
                f"→ 예측 테이블 단위/버전 또는 DART 분기화 점검 필요")
        return self

    def _sum_cols(self, keys) -> pd.Series:
        total = pd.Series(0.0, index=self.wide.index)
        found = []
        for k in keys:
            if k in self.wide.columns:
                total = total + self.wide[k].fillna(0.0)
                found.append(k)
        return total, found

    # ── 2. 계수 추정 ─────────────────────────────────────────
    def _ratio_coef(self, target: pd.Series, name: str) -> float:
        """target ≈ β × sales : OLS 채택 조건 미달 시 winsorized median."""
        df = pd.concat([self.sales.rename("s"), target.rename("t")], axis=1).dropna()
        df = df[df["s"] > 0]
        if df.empty:
            self._note(f"{name}: 데이터 없음 → 0")
            return 0.0
        slope, r2, n = _ols_ratio(df["s"], df["t"])
        if not np.isnan(slope) and r2 >= OLS_MIN_R2 and slope >= 0:
            self._log(f"{name} = {slope:.4f} (OLS, R²={r2:.2f}, n={n})")
            return slope
        med = float(_winsorize(df["t"] / df["s"]).median())
        self._log(f"{name} = {med:.4f} (median fallback, OLS R²={r2:.2f})")
        return max(med, 0.0)

    def estimate(self):
        w = self.wide

        # 실효세율
        if {"pretax_income", "tax_expense"}.issubset(w.columns):
            df = w[["pretax_income", "tax_expense"]].dropna()
            df = df[df["pretax_income"] > 0]
            if len(df) >= 4:
                self.tax = float((df["tax_expense"] / df["pretax_income"])
                                 .clip(0, 0.40).median())
            else:
                self.tax = TAX_FALLBACK
                self._note(f"실효세율 표본부족 → fallback {TAX_FALLBACK:.0%}")
        else:
            self.tax = TAX_FALLBACK
            self._note(f"세금 데이터 없음 → fallback {TAX_FALLBACK:.0%}")
        self._log(f"실효세율 = {self.tax:.2%}")

        # OPM (영업이익/매출)
        self.opm = self._ratio_coef(w["operating_income"], "OPM")
        if self.opm <= 0:
            df = pd.concat([self.sales.rename("s"),
                            w["operating_income"].rename("t")], axis=1).dropna()
            self.opm = float(_winsorize(df["t"] / df["s"]).median())
            self._note(f"OPM 음수 계열 → median {self.opm:.4f} 사용")

        # D&A / CapEx (CF 기반, 누적 차분 완료된 분기치)
        da_series, da_found = self._sum_cols(["da_cf", "intangible_amort_cf"])
        self.alpha_da = self._ratio_coef(da_series.replace(0, np.nan).dropna(), "D&A/Sales") \
            if da_found else 0.0
        capex_series, cx_found = self._sum_cols(["capex_tangible", "capex_intangible"])
        self.beta_capex = self._ratio_coef(capex_series.abs().replace(0, np.nan).dropna(),
                                           "CapEx/Sales") if cx_found else 0.0
        if not da_found or self.alpha_da == 0:
            self._note("⚠️ D&A 계수=0 — FCFF가 과소평가됩니다. "
                       "diagnose_accounts()로 CF 계정명을 확인해 FIELD_MAP 보강 필요")
        if not cx_found:
            self._note("CF CapEx 항목 없음 → β=0 (FCFF 낙관적 — 주의)")

        # NWC (Damodaran 영업운전자본 → 부채측 없으면 legacy proxy)
        op_ca, ca_found = self._sum_cols(self.NWC_ASSETS)
        op_cl, cl_found = self._sum_cols(self.NWC_LIABS)
        if ca_found and cl_found:
            nwc = op_ca - op_cl
            self.nwc_method = "operating"
        elif ca_found:
            st, _ = self._sum_cols(["short_term_debt", "current_lt_debt", "lease_liab"])
            nwc = op_ca - st
            self.nwc_method = "legacy_proxy"
            self._note("영업부채 항목 없음 → legacy NWC proxy")
        else:
            nwc = pd.Series(dtype=float)
            self.nwc_method = "none"
            self._note("NWC 항목 없음 → ΔNWC=0")

        if not nwc.dropna().empty:
            self.gamma_nwc = self._ratio_coef(nwc.dropna(), f"NWC/Sales({self.nwc_method})")
            self.last_nwc = float(nwc.dropna().iloc[-1])
        else:
            self.gamma_nwc, self.last_nwc = 0.0, 0.0
        return self

    # ── 3. WACC ──────────────────────────────────────────────
    def compute_wacc(self):
        w = self.wide
        # 부채 (최신 분기)
        debt_s, dk = self._sum_cols(self.DEBT_KEYS)
        self.debt = float(debt_s.dropna().iloc[-1]) if dk and not debt_s.dropna().empty else 0.0
        cash_s, ck = self._sum_cols(self.CASH_KEYS)
        self.cash_bal = float(cash_s.dropna().iloc[-1]) if ck and not cash_s.dropna().empty else 0.0
        self.net_debt = self.debt - self.cash_bal

        # 시가총액 (E)
        self.marketcap, self.mc_date = load_marketcap_latest(self.ticker)
        if self.marketcap is None:
            eq = w["total_equity"].dropna()
            self.marketcap = float(eq.iloc[-1]) if not eq.empty else np.nan
            self._note("시총 로드 실패 → 장부 자본총계로 대체 (Sanity 저하)")

        # Rd = 이자비용 / 평균 총부채, 하한 Rf
        rd = RD_DEFAULT
        if "interest_expense" in w.columns and self.debt > 0:
            ie = w["interest_expense"].abs()
            td_avg = (debt_s + debt_s.shift(1)) / 2
            ratio = (ie * 4 / td_avg).replace([np.inf, -np.inf], np.nan).dropna()
            if len(ratio) >= 4:
                rd = float(ratio.clip(0.005, 0.15).median())
        self.rd = max(rd, RF)                                        # v8 가드②

        # 베타
        self.beta = BETA_OVERRIDE.get(self.ticker, None)
        if self.beta is None:
            try:
                import FinanceDataReader as fdr
                px = fdr.DataReader(self.ticker, "2021-01-01")["Close"]
                mkt = fdr.DataReader("KS11", "2021-01-01")["Close"]
                r = pd.concat([px, mkt], axis=1).dropna().resample("W").last().pct_change().dropna()
                cov = np.cov(r.iloc[:, 0], r.iloc[:, 1])
                self.beta = float(cov[0, 1] / cov[1, 1])
            except Exception as e:
                self.beta = BETA_DEFAULT
                self._note(f"베타 산출 실패({type(e).__name__}) → {BETA_DEFAULT}")
        self.re = RF + self.beta * ERP

        E, D = self.marketcap, max(self.debt, 0.0)
        V = E + D if np.isfinite(E) else None
        wacc = (E / V * self.re + D / V * self.rd * (1 - self.tax)) if V else self.re
        self.wacc = float(np.clip(wacc, max(WACC_FLOOR_ABS, RF + 0.01), WACC_CAP))  # 가드②
        self._log(f"WACC={self.wacc:.2%} (Re={self.re:.2%} β={self.beta:.2f} "
                  f"Rd={self.rd:.2%} E={E/1e12:.1f}조 D={D/1e12:.1f}조)")
        return self

    # ── 4. FCFF 예측 & DCF ───────────────────────────────────
    def value(self):
        # ★ v8 방식: prev_nwc = γ × 마지막 실측 매출 (실측 NWC 레벨 시드는 첫 분기 ΔNWC 스파이크 유발)
        rows = []
        prev_nwc = self.gamma_nwc * float(self.sales.iloc[-1])
        for dt_, s in self.fc_sales.items():
            nopat = s * self.opm * (1 - self.tax)
            da    = s * self.alpha_da
            capex = s * self.beta_capex
            nwc   = s * self.gamma_nwc
            dnwc  = nwc - prev_nwc
            prev_nwc = nwc
            fcff = nopat + da - capex - dnwc
            rows.append({"date": dt_, "sales": s, "nopat": nopat, "da": da,
                         "capex": capex, "dnwc": dnwc, "fcff": fcff})
        self.proj = pd.DataFrame(rows).set_index("date")

        # 할인 (분기 중간시점 관행 생략 — 분기말 기준)
        qw = (1 + self.wacc) ** 0.25
        disc = np.array([qw ** -(i + 1) for i in range(len(self.proj))])
        pv_explicit = float((self.proj["fcff"].values * disc).sum())

        # Terminal (v8 가드③)
        g_term = min(GDP_GROWTH, RF)
        wacc_tv = max(self.wacc, g_term + MIN_TV_SPREAD)
        fcff_ann = float(self.proj["fcff"].iloc[-4:].sum())
        if fcff_ann <= 0:
            self._note("연간화 FCFF ≤ 0 → TV=0 (보수적)")
            tv = 0.0
        else:
            tv = fcff_ann * (1 + g_term) / (wacc_tv - g_term)
            cap = TV_FCFF_MULT_CAP * fcff_ann
            if tv > cap:
                self._note(f"TV {tv/fcff_ann:.0f}배 > {TV_FCFF_MULT_CAP:.0f}배 상한 → cap")
                tv = cap
        pv_tv = tv * float(disc[-1])

        self.ev = pv_explicit + pv_tv
        self.equity_value = self.ev - self.net_debt
        self.upside = (self.equity_value / self.marketcap - 1) \
            if np.isfinite(self.marketcap) and self.marketcap > 0 else np.nan

        self._log(f"PV(FCFF)={pv_explicit/1e12:.2f}조  PV(TV)={pv_tv/1e12:.2f}조  "
                  f"EV={self.ev/1e12:.2f}조  NetDebt={self.net_debt/1e12:.2f}조")
        self._log(f"지분가치={self.equity_value/1e12:.2f}조  "
                  f"시총={self.marketcap/1e12:.2f}조  Upside={self.upside:+.1%}")
        return self

    def run(self):
        return self.load().estimate().compute_wacc().value()

    def summary(self) -> dict:
        return {
            "ticker": self.ticker,
            "opm": self.opm, "tax": self.tax,
            "da_ratio": self.alpha_da, "capex_ratio": self.beta_capex,
            "nwc_ratio": self.gamma_nwc, "nwc_method": self.nwc_method,
            "beta": self.beta, "wacc": self.wacc,
            "ev_tril": self.ev / 1e12,
            "net_debt_tril": self.net_debt / 1e12,
            "equity_tril": self.equity_value / 1e12,
            "marketcap_tril": (self.marketcap or np.nan) / 1e12,
            "upside": self.upside,
            "fc_model": self.fc_model, "fc_date": self.fc_date,
            "notes": " | ".join(self.notes) if self.notes else "",
        }


print("[OK] DartFCFFModel 준비 완료")


[OK] DartFCFFModel 준비 완료


## Cell 6 · 실행

In [6]:
# ═══════════════════════════════════════════════════════════════
#  실행 — EXPORT_TICKERS 일괄 평가 → 요약 테이블
# ═══════════════════════════════════════════════════════════════
import traceback
import pandas as pd

results, failures = [], []

for tk in EXPORT_TICKERS:
    print("\n" + "=" * 70)
    print(f"▶ {tk} 평가 시작")
    print("=" * 70)
    try:
        m = DartFCFFModel(tk).run()
        results.append(m.summary())
        # 분기별 FCFF 전개 출력
        print("\n[FCFF 전개 (조원)]")
        print((m.proj / 1e12).round(3).to_string())
    except Exception as e:
        print(f"❌ {tk} 실패: {e}")
        if VERBOSE:
            traceback.print_exc()
        failures.append((tk, str(e)))

print("\n" + "=" * 70)
print(f"[결과 요약] 성공 {len(results)} / 실패 {len(failures)}")
print("=" * 70)

if results:
    summary_df = pd.DataFrame(results)
    pct = ["opm", "tax", "da_ratio", "capex_ratio", "nwc_ratio", "wacc", "upside"]
    disp = summary_df.copy()
    for c in pct:
        disp[c] = (disp[c] * 100).round(2)
    print(disp.drop(columns=["notes"]).to_string(index=False))
    for r in results:
        if r["notes"]:
            print(f"\n  [{r['ticker']} notes] {r['notes']}")

if failures:
    print("\n[실패 목록]")
    for tk, msg in failures:
        print(f"  {tk}: {msg[:120]}")



▶ 005930 평가 시작


C:\Users\82108\AppData\Local\Temp\ipykernel_25268\2775365695.py:155: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


[22:48:11][005930] wide shape=(44, 22) 기간=2015-03-31~2025-12-31  누적차분 적용: ['capex_tangible', 'capex_intangible']
[22:48:11][005930] 매출 actual=24Q forecast=8Q (Ensemble, 2026-06-01)
[22:48:11][005930] 실적 vs 예측 연결 구간:
  [actual 조원] {'2024-12': 75.79, '2025-03': 79.14, '2025-06': 74.57, '2025-09': 86.06}
  [forecast 조원] {'2026-06': 136.86, '2026-09': 149.67, '2026-12': 151.33, '2027-03': 156.46}
❌ 005930 실패: [005930] 실적-예측 단절: 최근 4분기 중앙값 77.5조 vs 예측 첫 분기 136.9조 (배율 1.77). → 예측 테이블 단위/버전 또는 DART 분기화 점검 필요

[결과 요약] 성공 0 / 실패 1

[실패 목록]
  005930: [005930] 실적-예측 단절: 최근 4분기 중앙값 77.5조 vs 예측 첫 분기 136.9조 (배율 1.77). → 예측 테이블 단위/버전 또는 DART 분기화 점검 필요


C:\Users\82108\AppData\Local\Temp\ipykernel_25268\3759188084.py:60: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(
Traceback (most recent call last):
  File "C:\Users\82108\AppData\Local\Temp\ipykernel_25268\2895447880.py", line 14, in <module>
    m = DartFCFFModel(tk).run()
  File "C:\Users\82108\AppData\Local\Temp\ipykernel_25268\233438872.py", line 274, in run
    return self.load().estimate().compute_wacc().value()
  File "C:\Users\82108\AppData\Local\Temp\ipykernel_25268\233438872.py", line 82, in load
    raise ValueError(
ValueError: [005930] 실적-예측 단절: 최근 4분기 중앙값 77.5조 vs 예측 첫 분기 136.9조 (배율 1.77). → 예측 테이블 단위/버전 또는 DART 분기화 점검 필요
